In [2]:
import pandas as pd
!pip install -q faiss-cpu sentence-transformers
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

kb = (
    train['prompt'].fillna('') + ' ' +
    train['A'].fillna('') + ' ' +
    train['B'].fillna('') + ' ' +
    train['C'].fillna('')
).tolist()

model = SentenceTransformer('all-MiniLM-L6-v2')

kb_embeddings = model.encode(
    kb,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)

index = faiss.IndexFlatIP(kb_embeddings.shape[1])
index.add(kb_embeddings)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 79.4 MB/s eta 0:00:00:00:0100:01


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load the training data from the correct path
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

print("Creating knowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating knowledge base
Loading embedding model and creating index


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge base successfully created


# Q.1

In [18]:
from transformers import pipeline

classifier = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli"
)

sample = train.loc[150]

question = str(sample["prompt"])

candidate_options = [
    str(sample[col])
    for col in ["A", "B", "C", "D", "E"]
]

prediction = classifier(question, candidate_options)

actual_option = sample["answer"]
actual_text = str(sample[actual_option])

confidence = next(
    (
        score
        for label, score in zip(
            prediction["labels"],
            prediction["scores"]
        )
        if label == actual_text
    ),
    0.0,
)

print(
    f"The predicted probability score assigned to the ground-truth correct option is: {confidence:.3f}"
)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

The predicted probability score assigned to the ground-truth correct option is: 0.384


# Q.2

In [19]:
import numpy as np

query = train.loc[150, "prompt"]

vector = model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)

TOP_K = 10

_, retrieved = index.search(
    vector.reshape(1, -1),
    TOP_K
)

expected_index = 150

ranking = None

for position, kb_index in enumerate(retrieved.ravel(), start=1):

    if kb_index == expected_index:
        ranking = position
        break

if ranking is None:
    print(f"The true correct document (original KB index 150) was not found in the top {TOP_K} results.")
else:
    print(f"The true correct document (original KB index 150) was placed at rank: {ranking}")

The true correct document (original KB index 150) was placed at rank: 10


# Q.3

In [20]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

candidate_ids = retrieved.ravel()

candidate_docs = list(
    map(kb.__getitem__, candidate_ids)
)

pairs = [
    (query, document)
    for document in candidate_docs
]

scores = reranker.predict(pairs)

ordered = sorted(
    zip(candidate_ids, scores),
    key=lambda item: item[1],
    reverse=True
)

correct_doc = 150

position = next(
    (
        idx + 1
        for idx, (doc_id, _) in enumerate(ordered)
        if doc_id == correct_doc
    ),
    None,
)

if position is None:
    print("The true correct document (original KB index 150) was not found in the top 10 results after reranking.")
else:
    print(f"The Cross-Encoder placed the true correct document (original KB index 150) at rank: {position}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The Cross-Encoder placed the true correct document (original KB index 150) at rank: 1


# Q.4

In [21]:
from transformers import AutoTokenizer
import numpy as np

question = train.loc[42, "prompt"]

embedding = model.encode(
    question,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)

TOP_K = 5

_, retrieved_ids = index.search(
    embedding.reshape(1, -1),
    TOP_K
)

context = " ".join(
    kb[idx]
    for idx in retrieved_ids.ravel()
)

rag_input = f"Context: {context}\nQuestion: {question}"

bert_tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

encoded = bert_tokenizer(
    rag_input,
    return_tensors="pt",
    truncation=False
)

print(
    f"The total number of tokens generated is: {encoded['input_ids'].size(1)}"
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The total number of tokens generated is: 216


# Q.5

In [22]:
reference_context = kb[150]

augmented_prompt = (
    f"Context: {reference_context}\n"
    f"Question: {question}"
)

prediction_with_rag = classifier(
    augmented_prompt,
    candidate_options
)

rag_score = next(
    (
        value
        for label, value in zip(
            prediction_with_rag["labels"],
            prediction_with_rag["scores"]
        )
        if label == actual_text
    ),
    0.0,
)

print(
    f"The new predicted probability score of the ground-truth correct option using RAG is: {rag_score:.3f}"
)

The new predicted probability score of the ground-truth correct option using RAG is: 0.984


# Q.6

In [23]:
incorrect_context = kb[999]

misleading_prompt = (
    f"Context: {incorrect_context}\n"
    f"Question: {question}"
)

prediction_bad_context = classifier(
    misleading_prompt,
    candidate_options
)

adversarial_score = next(
    (
        value
        for label, value in zip(
            prediction_bad_context["labels"],
            prediction_bad_context["scores"]
        )
        if label == actual_text
    ),
    0.0,
)

print(
    f"The new predicted probability score of the ground-truth correct option using an adversarial RAG context is: {adversarial_score:.3f}"
)

The new predicted probability score of the ground-truth correct option using an adversarial RAG context is: 0.274


# Q.7

In [16]:
import numpy as np

if "correct_answer_text" not in train_df.columns:
    train_df["correct_answer_text"] = train_df.apply(
        lambda sample: sample[sample["answer"]],
        axis=1
    )

TOP_K = 5
samples_to_test = 100
successful_hits = 0

for sample in train_df.head(samples_to_test).itertuples(index=False):

    question = sample.prompt
    expected_answer = sample.correct_answer_text

    vector = model.encode(
        question,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    _, retrieved_ids = index.search(vector.reshape(1, -1), TOP_K)

    retrieved_context = list(map(kb.__getitem__, retrieved_ids[0]))

    if any(expected_answer in passage for passage in retrieved_context):
        successful_hits += 1

hit_rate = (successful_hits / samples_to_test) * 100

print(f"Hit Rate@{TOP_K}: {hit_rate:.1f}%")

Hit Rate@5: 73.0%


# Q.8

In [17]:
import numpy as np

def average_precision_at_k(true_labels, predicted_labels, k=3):

    predicted_labels = predicted_labels[:k]

    score = 0.0
    correct = 0

    for rank, label in enumerate(predicted_labels, start=1):

        if label in true_labels and label not in predicted_labels[:rank-1]:
            correct += 1
            score += correct / rank

    if len(true_labels) == 0:
        return 0.0

    return score / min(len(true_labels), k)


evaluation_scores = []
retrieve_k = 5

for sample in train_df.head(20).itertuples(index=False):

    query = sample.prompt

    option_mapping = {
        "A": sample.A,
        "B": sample.B,
        "C": sample.C,
        "D": sample.D,
        "E": sample.E
    }

    embeddings = model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    _, retrieved = index.search(
        embeddings.reshape(1, -1),
        retrieve_k
    )

    retrieved_context = [kb[i] for i in retrieved[0]]

    pair_inputs = [(query, context) for context in retrieved_context]
    relevance_scores = cross_encoder.predict(pair_inputs)

    best_context = retrieved_context[int(np.argmax(relevance_scores))]

    rag_prompt = f"Context: {best_context}\nQuestion: {query}"

    prediction = zs(
        rag_prompt,
        list(option_mapping.values())
    )

    reverse_lookup = {v: k for k, v in option_mapping.items()}

    ranked_letters = []

    for option_text in prediction["labels"]:
        if option_text in reverse_lookup:
            ranked_letters.append(reverse_lookup[option_text])

        if len(ranked_letters) == 3:
            break

    evaluation_scores.append(
        average_precision_at_k(
            [sample.answer],
            ranked_letters,
            k=3
        )
    )

print(f"Average MAP@3 : {np.mean(evaluation_scores):.3f}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Average MAP@3 : 0.917
